# OB-SLIP workflow
This notebook shows a runnable OB-SLIP smoke test using the repository skeleton and synthetic data.

In [2]:
from pathlib import Path
import sys

import numpy as np

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import load_config
from src.preprocessing import percentile_normalize
from src.spectral_indices import compute_ndvi, compute_ndre, compute_brightness
from src.terrain_weighting import terrain_weight_from_config
from src.fusion_score import compute_fusion
from src.obia_postprocessing import otsu_threshold, apply_mmu

cfg = load_config(repo_root / 'config' / 'example_config.yml')
print('Loaded config sample:')
print(cfg)

# Synthetic smoke test so the notebook runs without proprietary data.
rng = np.random.default_rng(42)
rows, cols = 32, 32
pre = rng.uniform(0.15, 0.65, size=(5, rows, cols)).astype('float32')
post = pre.copy()
post[4] = np.clip(post[4] + 0.08, 0, 1)
post[2] = np.clip(post[2] - 0.06, 0, 1)
post[3] = np.clip(post[3] - 0.04, 0, 1)

pre_ndvi = compute_ndvi(pre[4], pre[2])
post_ndvi = compute_ndvi(post[4], post[2])
pre_ndre = compute_ndre(pre[4], pre[3])
post_ndre = compute_ndre(post[4], post[3])
pre_brightness = compute_brightness(pre)
post_brightness = compute_brightness(post)

dndvi = percentile_normalize(post_ndvi - pre_ndvi, lower=cfg['normalization']['lower_pct'], upper=cfg['normalization']['upper_pct'])
dndre = percentile_normalize(post_ndre - pre_ndre, lower=cfg['normalization']['lower_pct'], upper=cfg['normalization']['upper_pct'])
dbrightness = percentile_normalize(post_brightness - pre_brightness, lower=cfg['normalization']['lower_pct'], upper=cfg['normalization']['upper_pct'])

slope = np.linspace(0, 60, rows * cols, dtype='float32').reshape(rows, cols)
wterrain = terrain_weight_from_config(slope, cfg['terrain'])

score = compute_fusion(
    dndvi,
    dndre,
    dbrightness,
    weights=cfg['fusion']['weights'],
    wterrain=wterrain,
    wpost=1.0,
)

mask, threshold_value = otsu_threshold(score)
filtered = apply_mmu(mask, mmu_pixels=cfg['thresholding']['mmu_pixels'])

print('\nSynthetic smoke test summary:')
print(f'  dNDVI range: {dndvi.min():.3f} to {dndvi.max():.3f}')
print(f'  dNDRE range: {dndre.min():.3f} to {dndre.max():.3f}')
print(f'  dBrightness range: {dbrightness.min():.3f} to {dbrightness.max():.3f}')
print(f'  Fusion threshold: {threshold_value:.4f}')
print(f'  Pixels above threshold: {int(mask.sum())}')
print(f'  Pixels after MMU: {int(filtered.sum())}')

Loaded config sample:
{'inputs': {'pre_image': '/path/to/pre_event.tif', 'post_image': '/path/to/post_event.tif', 'dem': '/path/to/dem.tif', 'output_dir': './outputs'}, 'normalization': {'lower_pct': 2, 'upper_pct': 98, 'clip_range': [0, 1]}, 'fusion': {'weights': {'dndvi': 0.7, 'dndre': 0.2, 'dbrightness': 0.1}, 'apply_terrain_weight': True, 'apply_post_mask_weight': True}, 'terrain': {'slope_thresholds': {'low': 1.0, 'ramp_min': 1.0, 'ramp_max': 20.0, 'flat_max': 45.0}, 'ramp_min_weight': 0.2, 'ramp_max_weight': 1.0, 'high_slope_weight': 0.85}, 'thresholding': {'method': 'otsu', 'mmu_pixels': 9}, 'validation': {'buffer_m': 0, 'iou_threshold': 0.3}}

Synthetic smoke test summary:
  dNDVI range: 0.000 to 1.000
  dNDRE range: 0.000 to 1.000
  dBrightness range: 0.000 to 1.000
  Fusion threshold: 0.3372
  Pixels above threshold: 331
  Pixels after MMU: 263



Notes:
- Replace placeholder paths in `config/example_config.yml` with your own data when moving from the smoke test to real runs.
- Do not include proprietary imagery in the repo.

In [7]:
from pathlib import Path
from src.utils import load_config

def main():
    root = Path(__file__).resolve().parent
    config_path = root / "config" / "example_config.yml"

    config = load_config(str(config_path))
    print("OB-SLIP pipeline initialized")

if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

In [8]:
git add run_obslip.py
git commit -m "Add runnable OB-SLIP entry script"
git push

SyntaxError: invalid syntax (466780003.py, line 1)